# Olist E-Commerce Data Cleaning & Transformation

## Objective

Transform the raw Olist e-commerce datasets into clean, consistent, and analytics-ready datasets.

The transformations in this phase are based on issues identified during data profiling. Raw source data will remain unchanged, and cleaned datasets will be written separately to `data/processed/`.

### Transformation Goals

- Standardize column names and correct inconsistent naming.
- Convert date and timestamp fields to appropriate datetime types.
- Handle missing values based on their business meaning.
- Standardize categorical fields where necessary.
- Translate product categories from Portuguese to English.
- Create useful derived fields for later analysis.
- Validate transformed data before loading it into SQL.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Raw data:", RAW_DATA_DIR)
print("Processed data:", PROCESSED_DATA_DIR)

dataset_files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

# Reading with pandas, storing in datasets dictionary with name as key
datasets = {}

for name, filename in dataset_files.items():
    file_path = RAW_DATA_DIR / filename
    dataframe = pd.read_csv(file_path)
    datasets[name] = dataframe

    
for name, df in datasets.items():
    print(f"{name:<22} {df.shape}")

Raw data: d:\Documents\Career\Projects\retail-business-intelligence-platform\data\raw
Processed data: d:\Documents\Career\Projects\retail-business-intelligence-platform\data\processed
customers              (99441, 5)
geolocation            (1000163, 5)
order_items            (112650, 7)
payments               (103886, 5)
reviews                (99224, 7)
orders                 (99441, 8)
products               (32951, 9)
sellers                (3095, 4)
category_translation   (71, 2)


In [3]:
# Fix Spelling for some columns in products dataset

datasets["products"] = datasets["products"].rename(
    columns={
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length"
    }
)
print(datasets["products"].columns.tolist())

['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


## Datetime Standardization

Convert date and timestamp columns from strings into pandas datetime values so they can be used for time-based calculations, filtering, aggregation, and downstream SQL/Power BI analysis.

In [4]:
datetime_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "shipping_limit_date"
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

for dataset_name, columns in datetime_columns.items():

    for column in columns:
        datasets[dataset_name][column] = pd.to_datetime(
            datasets[dataset_name][column]
        )

for dataset_name, columns in datetime_columns.items():

    print(f"\n{dataset_name.upper()}")

    for column in columns:
        print(
            column,
            "->",
            datasets[dataset_name][column].dtype
        )


ORDERS
order_purchase_timestamp -> datetime64[us]
order_approved_at -> datetime64[us]
order_delivered_carrier_date -> datetime64[us]
order_delivered_customer_date -> datetime64[us]
order_estimated_delivery_date -> datetime64[us]

ORDER_ITEMS
shipping_limit_date -> datetime64[us]

REVIEWS
review_creation_date -> datetime64[us]
review_answer_timestamp -> datetime64[us]


## Missing-Value Handling

Handle missing values according to their business meaning and intended analytical use. Missing records are not automatically removed, since some null values represent optional information or incomplete lifecycle events rather than unusable data.

### Review Text

In [5]:
# Fill missing review text with descriptive values
datasets["reviews"]["review_comment_title"] = (datasets["reviews"]["review_comment_title"].fillna("No title"))

datasets["reviews"]["review_comment_message"] = (datasets["reviews"]["review_comment_message"].fillna("No comment"))

print(datasets["reviews"][["review_comment_title", "review_comment_message"]].isna().sum())

print(datasets["reviews"][["review_comment_title", "review_comment_message"]].head())

review_comment_title      0
review_comment_message    0
dtype: int64
  review_comment_title                             review_comment_message
0             No title                                         No comment
1             No title                                         No comment
2             No title                                         No comment
3             No title              Recebi bem antes do prazo estipulado.
4             No title  Parabéns lojas lannister adorei comprar pela I...


Missing review titles were replaced with `"No title"`, while missing review messages were replaced with `"No comment"`.

The fields were handled separately because a customer may provide one without providing the other. This preserves the distinction between an absent title and an absent written comment while retaining the review record and its numeric score.

### Products

In [6]:
datasets["products"]["product_category_name"] = (datasets["products"]["product_category_name"].fillna("unknown"))
print("Missing product categories:",datasets["products"]["product_category_name"].isna().sum())
print(datasets["products"][datasets["products"]["product_category_name"]=="unknown"].head())

Missing product categories: 0
                           product_id product_category_name  \
105  a41e356c76fab66334f36de622ecbd3a               unknown   
128  d8dee61c2034d6d075997acef1870e9b               unknown   
145  56139431d72cd51f19eb9f7dae4d1617               unknown   
154  46b48281eb6d663ced748f324108c733               unknown   
197  5fb61f482620cb672f5e586bb132eae9               unknown   

     product_name_length  product_description_length  product_photos_qty  \
105                  NaN                         NaN                 NaN   
128                  NaN                         NaN                 NaN   
145                  NaN                         NaN                 NaN   
154                  NaN                         NaN                 NaN   
197                  NaN                         NaN                 NaN   

     product_weight_g  product_length_cm  product_height_cm  product_width_cm  
105             650.0               17.0              

We'll leave these numeric metadata fields `NULL`: `product_name_length`, `product_description_length`, `product_photos_qty`, `product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm` but lets add a flag that will help us filter these missing values later.

In [7]:
datasets["products"]["product_metadata_missing"] = (datasets["products"][
    [
        "product_name_length",
        "product_description_length",
        "product_photos_qty"
    ]
    ]
    .isna().all(axis=1)
)

measurement_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

datasets["products"]["product_measurements_missing"] = (datasets["products"][measurement_columns]
    .isna().all(axis=1) # checks whether every selected column is missing (NaN) for each row. # Returns “Which rows have ALL of these columns missing?”
)

print(datasets["products"]["product_metadata_missing"].value_counts())

print(datasets["products"]["product_measurements_missing"].value_counts())

product_metadata_missing
False    32341
True       610
Name: count, dtype: int64
product_measurements_missing
False    32949
True         2
Name: count, dtype: int64


### Orders

A missing timestamp has business meaning, so we should preserve it as NaT and make the records easier to identify with flags.

In [8]:
# Flag missing order lifecycle timestamps
datasets["orders"]["approval_date_missing"] = (datasets["orders"]["order_approved_at"].isna())

datasets["orders"]["carrier_date_missing"] = (datasets["orders"]["order_delivered_carrier_date"].isna())

datasets["orders"]["delivery_date_missing"] = (datasets["orders"]["order_delivered_customer_date"].isna())

A delivered order with missing timestamps can be a really useful flag to keep track of. This could help us identify the issues with deliveries

In [9]:
datasets["orders"]["delivered_order_data_incomplete"] = (
    (datasets["orders"]["order_status"] == "delivered")
    &
    (
        datasets["orders"][
            [
                "order_approved_at",
                "order_delivered_carrier_date",
                "order_delivered_customer_date"
            ]
        ]
        .isna()
        .any(axis=1)
    )
)

print(datasets["orders"]["delivered_order_data_incomplete"].value_counts())

delivered_order_data_incomplete
False    99418
True        23
Name: count, dtype: int64


## Product Category Translation

Join the product dataset with the provided category translation table to add English product category names for downstream reporting and visualization.

In [10]:
# print(datasets["category_translation"].head())

# Match the 2 datasets with the common column `product_category_name`, 'left' keeps every product even if no value.
datasets["products"] = datasets["products"].merge(datasets["category_translation"],on="product_category_name",how="left")

print(datasets["products"][["product_category_name","product_category_name_english"]].head(10))

print("Missing English categories:",datasets["products"]["product_category_name_english"].isna().sum())

# loc[rows, columns] lets u select specific rows and columns in a dataset
# in this case we are selecting rows that have `product_category_name_english` as null
# Then selecting the `product_category_name` column
# Then value_counts counts how many times each category occurs.
print(datasets["products"].loc[datasets["products"]["product_category_name_english"].isna(),"product_category_name"].value_counts())

   product_category_name product_category_name_english
0             perfumaria                     perfumery
1                  artes                           art
2          esporte_lazer                sports_leisure
3                  bebes                          baby
4  utilidades_domesticas                    housewares
5  instrumentos_musicais           musical_instruments
6             cool_stuff                    cool_stuff
7       moveis_decoracao               furniture_decor
8       eletrodomesticos               home_appliances
9             brinquedos                          toys
Missing English categories: 623
product_category_name
unknown                                          610
portateis_cozinha_e_preparadores_de_alimentos     10
pc_gamer                                           3
Name: count, dtype: int64


The category translation successfully converted most Portuguese product categories into English. However, 623 products did not receive an English category after the merge.

Of the 623 unmatched products:
- 610 had already been assigned the category `unknown` because their original product category was missing.
- 10 belonged to `portateis_cozinha_e_preparadores_de_alimentos`.
- 3 belonged to `pc_gamer`.

This shows that the missing English categories are not caused only by missing product data. Two valid Portuguese categories also do not have a matching entry in the provided category translation dataset and will require separate handling. We'll manually fix the 13 protugese categories without a translation and add unknown for the remaining 610.

In [11]:
# Select the rows where `product_category_name` is `portateis_cozinha_e_preparadores_de_alimentos`
# And the `product_category_name_english` columns
# and == them to `portable_kitchen_and_food_preparation`
datasets["products"].loc[datasets["products"]["product_category_name"] == "portateis_cozinha_e_preparadores_de_alimentos", "product_category_name_english"] = "portable_kitchen_and_food_preparation"

datasets["products"].loc[datasets["products"]["product_category_name"] == "pc_gamer", "product_category_name_english"] = "pc_gaming"

# Then we fill in the missing null ones with unknown
datasets["products"]["product_category_name_english"] = datasets["products"]["product_category_name_english"].fillna("unknown")

print("Missing English categories:", datasets["products"]["product_category_name_english"].isna().sum())

Missing English categories: 0


## Derived Order Fields

Create additional fields from the cleaned order timestamps to support delivery performance and time-based analysis in SQL and Power BI. This will eventually let us create Power BI metrics like late delivery rate, average delivery time, and average days early/late.

In [12]:
# First field: delivery time. We want to know how many days passed between the customer placing an order and actually receiving it.
datasets["orders"]["delivery_days"] = (datasets["orders"]["order_delivered_customer_date"] - datasets["orders"]["order_purchase_timestamp"]).dt.days
print(datasets["orders"][["order_purchase_timestamp", "order_delivered_customer_date", "delivery_days"]].head(7))

# Second field: days early or late comparing delivery date to estimated delivery date
datasets["orders"]["delivery_vs_estimate_days"] = (datasets["orders"]["order_delivered_customer_date"] - datasets["orders"]["order_estimated_delivery_date"]).dt.days
print(datasets["orders"][["order_delivered_customer_date", "order_estimated_delivery_date", "delivery_vs_estimate_days"]].head(7))

  order_purchase_timestamp order_delivered_customer_date  delivery_days
0      2017-10-02 10:56:33           2017-10-10 21:25:13            8.0
1      2018-07-24 20:41:37           2018-08-07 15:27:45           13.0
2      2018-08-08 08:38:49           2018-08-17 18:06:29            9.0
3      2017-11-18 19:28:06           2017-12-02 00:28:42           13.0
4      2018-02-13 21:18:39           2018-02-16 18:17:02            2.0
5      2017-07-09 21:57:05           2017-07-26 10:57:55           16.0
6      2017-04-11 12:22:08                           NaT            NaN
  order_delivered_customer_date order_estimated_delivery_date  \
0           2017-10-10 21:25:13                    2017-10-18   
1           2018-08-07 15:27:45                    2018-08-13   
2           2018-08-17 18:06:29                    2018-09-04   
3           2017-12-02 00:28:42                    2017-12-15   
4           2018-02-16 18:17:02                    2018-02-26   
5           2017-07-26 10:57:55   

### Delivery Status Classification

Classify orders based on whether they were delivered before, on, or after their estimated delivery date. Orders without an actual delivery date are classified separately.
We have four possibilities:

Negative delivery_vs_estimate_days → "Early"

0 → "On Time"

Positive → "Late"

Missing → "Missing Delivery Data"

In [13]:
conditions = [
    datasets["orders"]["order_delivered_customer_date"].isna(),
    datasets["orders"]["delivery_vs_estimate_days"] < 0,
    datasets["orders"]["delivery_vs_estimate_days"] == 0,
    datasets["orders"]["delivery_vs_estimate_days"] > 0
]

choices = ["Missing Delivery Data", "Early", "On Time", "Late"]

# np.select(conditions, choices, default) = check several conditions and assign the corresponding choice. Kinda like bunch of if statements. else:"Unknown"
datasets["orders"]["delivery_status"] = np.select(conditions, choices, default="Unknown")

print(datasets["orders"]["delivery_status"].value_counts())
print("Missing delivery statuses:", (datasets["orders"]["delivery_status"] == "Unknown").sum())

delivery_status
Early                    88649
Late                      6535
Missing Delivery Data     2965
On Time                   1292
Name: count, dtype: int64
Missing delivery statuses: 0


## Final Data Validation

Perform final quality checks to confirm that the cleaning and transformation process produced consistent, analytics-ready datasets before exporting them for SQL analysis.

In [14]:
expected_row_counts = {
    "customers": 99441,
    "geolocation": 1000163,
    "order_items": 112650,
    "payments": 103886,
    "reviews": 99224,
    "orders": 99441,
    "products": 32951,
    "sellers": 3095,
    "category_translation": 71
}

for name, expected_count in expected_row_counts.items():
    actual_count = len(datasets[name])
    print(name, "| Expected:", expected_count, "| Actual:", actual_count, "| Match:", actual_count == expected_count)

customers | Expected: 99441 | Actual: 99441 | Match: True
geolocation | Expected: 1000163 | Actual: 1000163 | Match: True
order_items | Expected: 112650 | Actual: 112650 | Match: True
payments | Expected: 103886 | Actual: 103886 | Match: True
reviews | Expected: 99224 | Actual: 99224 | Match: True
orders | Expected: 99441 | Actual: 99441 | Match: True
products | Expected: 32951 | Actual: 32951 | Match: True
sellers | Expected: 3095 | Actual: 3095 | Match: True
category_translation | Expected: 71 | Actual: 71 | Match: True


In [15]:
print("Missing English product categories:", datasets["products"]["product_category_name_english"].isna().sum())
print("Missing delivery statuses:", datasets["orders"]["delivery_status"].isna().sum())
print("Unknown delivery statuses:", (datasets["orders"]["delivery_status"] == "Unknown").sum())
print("Product metadata missing flags:", datasets["products"]["product_metadata_missing"].sum())
print("Product measurement missing flags:", datasets["products"]["product_measurements_missing"].sum())
print("Delivered orders with incomplete lifecycle data:", datasets["orders"]["delivered_order_data_incomplete"].sum())

Missing English product categories: 0
Missing delivery statuses: 0
Unknown delivery statuses: 0
Product metadata missing flags: 610
Product measurement missing flags: 2
Delivered orders with incomplete lifecycle data: 23


## Export Cleaned Datasets

Export the cleaned and transformed datasets to the processed data directory for use in the SQL database and subsequent analytics stages.

In [16]:
for name, dataframe in datasets.items():
    output_path = PROCESSED_DATA_DIR / f"{name}_cleaned.csv"
    dataframe.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

Saved: d:\Documents\Career\Projects\retail-business-intelligence-platform\data\processed\customers_cleaned.csv
Saved: d:\Documents\Career\Projects\retail-business-intelligence-platform\data\processed\geolocation_cleaned.csv
Saved: d:\Documents\Career\Projects\retail-business-intelligence-platform\data\processed\order_items_cleaned.csv
Saved: d:\Documents\Career\Projects\retail-business-intelligence-platform\data\processed\payments_cleaned.csv
Saved: d:\Documents\Career\Projects\retail-business-intelligence-platform\data\processed\reviews_cleaned.csv
Saved: d:\Documents\Career\Projects\retail-business-intelligence-platform\data\processed\orders_cleaned.csv
Saved: d:\Documents\Career\Projects\retail-business-intelligence-platform\data\processed\products_cleaned.csv
Saved: d:\Documents\Career\Projects\retail-business-intelligence-platform\data\processed\sellers_cleaned.csv
Saved: d:\Documents\Career\Projects\retail-business-intelligence-platform\data\processed\category_translation_cleaned